# Session 13 — Your turn: Transformers beyond text

Count what a vision transformer sees, then do zero-shot classification the way CLIP does: pick the class whose text vector is most similar to the image vector.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np
from aij_check import check, close, summary

## Task 1: Patches are tokens

Write `vit_tokens(image_size, patch_size)` returning the number of patch tokens (`(image_size // patch_size)**2`) plus one class token, and `patch_embedding_params(patch_size, channels, width)` for the linear layer that turns one patch into a vector (weights plus bias).
Use them for a 224 x 224 RGB image with 16 x 16 patches and width 768: `n_tokens`, `embed_params`. Then `n_tokens_small` for 32 x 32 patches, and `attention_pairs_ratio`, how many times more token pairs the 16 x 16 version has.

In [ ]:
def vit_tokens(image_size, patch_size):
    ...


def patch_embedding_params(patch_size, channels, width):
    ...


n_tokens = ...
embed_params = ...
n_tokens_small = ...
attention_pairs_ratio = ...

In [ ]:
check("196 patches and a class token: 197 tokens", lambda: n_tokens == 197)
check("the patch embedding has 590,592 parameters", lambda: embed_params == 590_592)
check("32 x 32 patches give 50 tokens", lambda: n_tokens_small == 50)
check("attention over 197 tokens costs about 15 times more pairs than over 50", lambda: close(attention_pairs_ratio, 15.5, 0.1))
summary()

## Task 2: Zero-shot classification by similarity

`text_vectors` holds one vector per class name (`classes`), `image_vectors` one per image, and `labels` the true class of each image. Normalise all vectors to length 1, compute the cosine similarity of each image to each class text, predict the most similar class (`predicted`), and store the `accuracy`.
Then give the same classification with a **prompt shift**: `text_vectors_shifted` is what you would get from a badly worded prompt; compute `accuracy_shifted` the same way.

In [ ]:
rng = np.random.default_rng(0)
classes = ["daisy", "rose", "tulip"]
text_vectors = rng.normal(size=(3, 16))
labels = rng.integers(0, 3, 90)
image_vectors = text_vectors[labels] * 0.6 + rng.normal(size=(90, 16))                    # an image is its class's direction plus noise
text_vectors_shifted = text_vectors + 2.0 * rng.normal(size=(3, 16))                      # a badly worded prompt moves the class vectors

def classify(images, texts):
    ...


predicted = ...
accuracy = ...
accuracy_shifted = ...

In [ ]:
check("one prediction per image", lambda: np.shape(predicted) == (90,))
check("with good text vectors the accuracy is well above chance (1/3)", lambda: accuracy > 0.6)
check("a badly worded prompt costs accuracy", lambda: accuracy_shifted < accuracy - 0.1)
check("classify uses cosine similarity: scaling the images does not change the answer", lambda: np.array_equal(classify(image_vectors * 5, text_vectors), predicted))
summary()